# Tome 11, Chapter 12: Log-correlated fields and the largest values (Python companion)

Exercises first, solutions at the end. The branching random walk and Bramson's $\frac32$, a log-correlated Fourier field and its multiplicative chaos, the maximum of the characteristic polynomial of a random unitary matrix and the $\frac34$ of Fyodorov, Hiary and Keating, the same maximum for the **Riemann zeta function** near height $10^6$, and a computer science application: **the height of random binary search trees**, which is a branching random walk in disguise. Requires `numpy` and `scipy`.

In [1]:
import math
import numpy as np
from scipy import stats, special
rng = np.random.default_rng(1112)
cstar = math.sqrt(2 * math.log(2))

def haar_angles(n):
    G = (rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))) / math.sqrt(2)
    Q, R = np.linalg.qr(G); d = np.diag(R)
    return np.angle(np.linalg.eigvals(Q * (d / abs(d))[None, :]))

def max_logZ(th, K):
    # maximum of log|det(I - e^{-i t} U)| over a grid of K points (Lemma 12.5 says K of order n suffices)
    grid = np.linspace(0, 2 * np.pi, K, endpoint=False); out = np.zeros(K)
    for s in range(0, len(th), 256):
        out += np.log(np.abs(1 - np.exp(1j * (th[s:s+256][:, None] - grid[None, :])))).sum(0)
    return out.max()

## Exercises

**P12.1 (★) Bramson's correction.** Simulate the maximum $M_n$ of the binary branching random walk for $n=6,8,\dots,20$ and compare its mean with $c^*n-\frac3{2c^*}\log n$, and the maximum of $2^n$ independent $N(0,n)$ variables with $c^*n-\frac1{2c^*}\log n$. Do the differences look bounded?

**P12.2 (★★) A log-correlated field.** Sample the random Fourier field $X_N$ of (12.1) on a grid with the fast Fourier transform. Check its covariance against Lemma 12.2 and Exercise 12.4, and compare the mean of $\max_\theta X_N(\theta)$ with $\log N-\frac34\log\log N$.

**P12.3 (★★) Multiplicative chaos.** For $\gamma=0.5,1,1.3$ and $N=2^6,\dots,2^{14}$, estimate $\mathbb EM_N(1)$ and $\mathbb EM_N(1)^2$ and compare with $1$ and with $\frac{\Gamma(1-\gamma^2/2)}{\Gamma(1-\gamma^2/4)^2}$ (Exercise 12.5). What fraction of the mass of $M_N$ sits on the $1\%$ of grid points where $X_N$ is largest? Compare $\gamma=1$ and $\gamma=1.9$.

**P12.4 (★★) The maximum of the characteristic polynomial.** For $n=16,\dots,512$, sample $\max_\theta\log|\det(I-e^{-i\theta}U)|$ and look at the law of $\max-(\log n-\frac34\log\log n)$: does its spread stay bounded, as Theorem 12.7 says?

**P12.5 (★★★) The zeta function on intervals of length $2\pi$.** Compute $\max\log|\zeta(\frac12+it)|$ over $400$ consecutive intervals $[T+2\pi k,T+2\pi(k+1)]$ with $T=10^6$, using the Riemann–Siegel function of Chapter 11, and compare the distribution of these maxima with that of the maximum of $\log|\det(I-e^{-i\theta}U)|$ for Haar $U\in U(n)$, $n=\log\frac T{2\pi}\approx12$.

**P12.6 (★★★) Computer science application: the height of a random binary search tree.** Insert $n$ distinct keys in uniformly random order into a binary search tree. Luc Devroye proved in 1986 that the height $H_n$ satisfies $H_n/\log n\to\alpha=4.31107\ldots$, the root $>2$ of $\alpha\log\frac{2e}\alpha=1$, and Bruce Reed in 2003 that $H_n=\alpha\log n-\beta\log\log n+O(1)$ with $\beta=\frac3{2\log(\alpha/2)}$, a Bramson correction. Simulate $H_n$ up to $n=2^{18}$ in time $O(n)$ per tree and compare.

## Solutions

### P12.1

**How to find it.** Build the walk level by level: the $2^k$ values at depth $k$ become $2^{k+1}$ values by adding independent Gaussians. For independent variables, sample the maximum directly: $P(\max\le x)=\Phi(x)^N$, so $\max=\Phi^{-1}(U^{1/N})$, computed through the upper tail for accuracy.

In [2]:
def brw_max(n):
    S = np.zeros(1)
    for _ in range(n): S = np.repeat(S, 2) + rng.standard_normal(2 * len(S))
    return S.max()

def indep_max(n, size):
    U = rng.random(size); tail = -np.expm1(np.log(U) / 2**n)    # 1 - U^{1/N}
    return math.sqrt(n) * stats.norm.isf(tail)

print("  n   E M_n    c*n - 3/(2c*) log n   diff    E max indep   c*n - 1/(2c*) log n   diff")
for n in range(6, 21, 2):
    reps = 300 if n <= 14 else 40
    m = np.mean([brw_max(n) for _ in range(reps)]); mi = indep_max(n, 4000).mean()
    b3 = cstar * n - 3 / (2 * cstar) * math.log(n); b1 = cstar * n - 1 / (2 * cstar) * math.log(n)
    print(f"{n:>3}   {m:6.3f}   {b3:19.3f}   {m - b3:5.2f}    {mi:10.3f}   {b1:19.3f}   {mi - b1:5.2f}")

  n   E M_n    c*n - 3/(2c*) log n   diff    E max indep   c*n - 1/(2c*) log n   diff
  6    4.900                 4.782    0.12         5.751                 6.304   -0.55
  8    6.988                 6.770    0.22         7.997                 8.536   -0.54
 10    9.056                 8.841    0.22        10.278                10.796   -0.52
 12   11.504                10.963    0.54        12.529                13.074   -0.54
 14   13.281                13.122    0.16        14.854                15.363   -0.51


 16   15.928                15.306    0.62        17.178                17.661   -0.48


 18   17.571                17.511    0.06        19.484                19.966   -0.48


 20   19.839                19.732    0.11        21.804                22.276   -0.47


For independent variables the difference settles near $-0.5$. For the tree it fluctuates between $0.1$ and $0.6$ without a trend (the large-$n$ rows use only $40$ runs each), while the maxima themselves grow from $5$ to $20$. With the independent correction $\frac1{2c^*}\log n$ instead of $\frac3{2c^*}\log n$, the tree's difference would drift down by $\frac1{c^*}\log n$, about $1.4$ over this range: the tree's maximum really is lower than that of independent variables, by a logarithm.

### P12.2

**How to find it.** $X_N(\theta_m)=\operatorname{Re}\sum_jG_jj^{-1/2}e^{-ij\theta_m}$ on the grid $\theta_m=\frac{2\pi m}K$ is the real part of a discrete Fourier transform. A grid with $K=8N$ points resolves the field.

In [3]:
def field(N, K=None):
    K = K or 8 * N
    c = np.zeros(K, dtype=complex); j = np.arange(1, N + 1)
    c[1:N + 1] = (rng.standard_normal(N) + 1j * rng.standard_normal(N)) / math.sqrt(2) / np.sqrt(j)
    return np.fft.fft(c).real

N = 1024; K = 8 * N; F = np.array([field(N) for _ in range(2000)])
print("variance: empirical %.4f, (1/2) H_N = %.4f" % (F.var(axis=0).mean(), 0.5 * np.sum(1 / np.arange(1, N + 1))))
print("  x        empirical cov   (1/2)log min(N,1/x)   -(1/2)log|2 sin(x/2)|")
for m in (1, 8, 64, 512, 2048, 4096):
    x = 2 * np.pi * m / K
    cov = np.mean(F[:, 0] * F[:, m])
    print(f"{x:8.4f}   {cov:13.4f}   {0.5 * math.log(min(N, 1 / x)):19.4f}   {-0.5 * math.log(abs(2 * math.sin(x / 2))):22.4f}")
print("   N     E max X_N   log N - (3/4) log log N   diff")
for N in (2**6, 2**8, 2**10, 2**12, 2**14):
    mx = np.mean([field(N).max() for _ in range(400)])
    p = math.log(N) - 0.75 * math.log(math.log(N))
    print(f"{N:>6}   {mx:9.3f}   {p:23.3f}   {mx - p:5.2f}")

variance: empirical 3.7440, (1/2) H_N = 3.7546
  x        empirical cov   (1/2)log min(N,1/x)   -(1/2)log|2 sin(x/2)|
  0.0008          3.9077                3.4657                   3.5865
  0.0061          2.7467                2.5468                   2.5468
  0.0491          1.6325                1.5071                   1.5071
  0.3927          0.6163                0.4674                   0.4706
  1.5708         -0.2952               -0.2258                  -0.1733
  3.1416         -0.3816               -0.5724                  -0.3466
   N     E max X_N   log N - (3/4) log log N   diff
    64       4.055                     3.090    0.97
   256       5.292                     4.260    1.03
  1024       6.459                     5.479    0.98


  4096       7.856                     6.729    1.13


 16384       9.128                     8.000    1.13


The covariance follows $\frac12\log\frac1x$ above the scale $\frac1N$ and saturates below it, as Lemma 12.2 says; at large distances it is $-\frac12\log|2\sin\frac x2|$, which becomes negative near $x=\pi$. The maximum grows like $\log N$ with a slowly varying correction, consistent with the $\frac34\log\log N$ (Ding, Roy and Zeitouni proved the $\frac34$ for a wide class of log-correlated Gaussian fields in 2017).

### P12.3

**How to find it.** $M_N(1)$ is the grid average of $e^{\gamma X_N-\gamma^2V_N/2}$ with $V_N=\frac12H_N$.

In [4]:
for gamma in (0.5, 1.0, 1.3):
    pred = special.gamma(1 - gamma**2 / 2) / special.gamma(1 - gamma**2 / 4)**2
    print(f"gamma = {gamma}: predicted E M(1)^2 = {pred:.4f}")
    for N in (2**6, 2**10, 2**14):
        V = 0.5 * np.sum(1 / np.arange(1, N + 1))
        masses = np.array([np.mean(np.exp(gamma * field(N) - gamma**2 * V / 2)) for _ in range(1000)])
        print(f"   N = {N:>6}:  mean {masses.mean():.4f}   second moment {np.mean(masses**2):.4f}")
for gamma in (1.0, 1.9):
    N = 2**14; V = 0.5 * np.sum(1 / np.arange(1, N + 1)); fr = []
    for _ in range(200):
        X = field(N); w = np.exp(gamma * X - gamma**2 * V / 2)
        top = X >= np.quantile(X, 0.99); fr.append(w[top].sum() / w.sum())
    print(f"gamma = {gamma}: fraction of the mass on the top 1% of points: {np.mean(fr):.3f}")

gamma = 0.5: predicted E M(1)^2 = 1.0071
   N =     64:  mean 1.0029   second moment 1.0122


   N =   1024:  mean 0.9992   second moment 1.0052


   N =  16384:  mean 0.9993   second moment 1.0067
gamma = 1.0: predicted E M(1)^2 = 1.1803
   N =     64:  mean 0.9925   second moment 1.1471
   N =   1024:  mean 0.9727   second moment 1.0661


   N =  16384:  mean 0.9767   second moment 1.0849
gamma = 1.3: predicted E M(1)^2 = 2.5234
   N =     64:  mean 0.9970   second moment 1.6992
   N =   1024:  mean 0.9694   second moment 1.6358


   N =  16384:  mean 0.9582   second moment 1.7476


gamma = 1.0: fraction of the mass on the top 1% of points: 0.443


gamma = 1.9: fraction of the mass on the top 1% of points: 0.925


The mean stays near $1$, but the empirical second moment falls short of the Gamma-function value of Exercise 12.5, more and more as $\gamma$ approaches $\sqrt2$ ($1.08$ against $1.18$ for $\gamma=1$, $1.75$ against $2.52$ for $\gamma=1.3$). The theorem is not wrong; the sample is too small. The total mass $M(1)$ has finite moments only of order $p<\frac4{\gamma^2}$, so for $\gamma=1.3$ even its fourth moment is infinite, and $1000$ samples rarely contain the large values that carry the second moment: the lesson of Problem P11.4 again. For $\gamma=1$, about $44\%$ of the mass already sits on the top $1\%$ of points; for $\gamma=1.9$, close to the critical value $2$, it is $92\%$: the chaos concentrates on the thick points (Exercise 12.6), and at criticality it would concentrate on the maximum.

### P12.4

**How to find it.** By Lemma 12.5, a grid of $8n$ points loses at most a bounded amount, and in practice almost nothing.

In [5]:
print("   n   mean of max   log n - (3/4) log log n   mean diff   std of diff")
for n in (16, 32, 64, 128, 256, 512):
    reps = 600 if n <= 128 else 150
    mx = np.array([max_logZ(haar_angles(n), 8 * n) for _ in range(reps)])
    p = math.log(n) - 0.75 * math.log(math.log(n))
    print(f"{n:>4}   {mx.mean():11.3f}   {p:23.3f}   {np.mean(mx - p):9.3f}   {np.std(mx - p):11.3f}")

   n   mean of max   log n - (3/4) log log n   mean diff   std of diff
  16         2.433                     2.008       0.425         0.470


  32         2.893                     2.534       0.359         0.510


  64         3.458                     3.090       0.368         0.609


 128         4.046                     3.667       0.379         0.640


 256         4.699                     4.260       0.439         0.686


 512         5.281                     4.865       0.416         0.713


The mean of the difference stays between $0.36$ and $0.44$ while $\log n$ more than doubles, and its standard deviation creeps up only slowly, from $0.47$ to $0.71$: the fluctuations of the maximum are of order $1$, as tightness says, and approach their limit slowly. Chhaibi, Madaule and Najnudel also identified the limit law up to a random shift: it has a Gumbel-like right tail with an extra factor, $P(\max-m_n>x)\approx Cxe^{-2x}$, the signature of log-correlated maxima.

### P12.5

**How to find it.** The Riemann–Siegel function $Z(t)$ has $|Z(t)|=|\zeta(\frac12+it)|$. A grid of step $0.005$ (more than $150$ points per zero) captures the maxima. For the matrices, $n=12$ is small enough to sample $20000$ of them.

In [6]:
def theta(t): return t / 2 * np.log(t / (2 * np.pi)) - t / 2 - np.pi / 8 + 1 / (48 * t) + 7 / (5760 * t**3)

def Zfun(t):
    t = np.atleast_1d(t).astype(float); a = np.sqrt(t / (2 * np.pi)); N = np.floor(a).astype(int); p = a - N
    out = np.empty_like(t)
    for s in range(0, len(t), 4000):
        tt, NN = t[s:s+4000], N[s:s+4000]; k = np.arange(1, NN.max() + 1)
        terms = np.cos(theta(tt)[:, None] - tt[:, None] * np.log(k)[None, :]) / np.sqrt(k)[None, :]
        terms[k[None, :] > NN[:, None]] = 0
        out[s:s+4000] = 2 * terms.sum(1)
    C0 = np.cos(2 * np.pi * (p * p - p - 1 / 16)) / np.cos(2 * np.pi * p)
    return out + (-1.0)**(N - 1) * (t / (2 * np.pi))**(-0.25) * C0

T = 1e6; nint = 400; h = 0.005
pts = int(round(2 * np.pi / h))
tt = T + h * np.arange(nint * pts)
zmax = np.log(np.abs(Zfun(tt))).reshape(nint, pts).max(axis=1)
n = round(math.log(T / (2 * np.pi)))
cue = np.array([max_logZ(haar_angles(n), 64 * n) for _ in range(20000)])
print("n =", n, "  log log T - (3/4) log log log T =", round(math.log(math.log(T)) - 0.75 * math.log(math.log(math.log(T))), 3))
print("             mean    std    10%    50%    90%")
for name, x in (("zeta", zmax), ("CUE(12)", cue)):
    q = np.quantile(x, [0.1, 0.5, 0.9])
    print(f"{name:>8}   {x.mean():6.3f}  {x.std():5.3f}  {q[0]:5.2f}  {q[1]:5.2f}  {q[2]:5.2f}")

n = 12   log log T - (3/4) log log log T = 1.902
             mean    std    10%    50%    90%
    zeta    2.221  0.468   1.67   2.23   2.79
 CUE(12)    2.187  0.437   1.66   2.14   2.77


The two distributions are remarkably close: the zeta maxima are higher by a few hundredths on average and slightly more spread out. Such a small difference is expected, and it is the same phenomenon as the moments of Problem P11.6: the random matrix model captures the statistics of the zeros, and the primes add a correction of order one, which for the moments was the arithmetic factor $a_m$. The theorems of Arguin, Bourgade and Radziwiłł show that both the matrices and the zeta function obey the same $\frac34$ law, with possibly different $O(1)$ terms.

### P12.6

**How to find it.** The binary search tree obtained by inserting keys in the order of a random permutation is the Cartesian tree of the sequence (key $i$, insertion time $t_i$): the root is the key inserted first, and each subtree is the same construction on an interval of keys. A stack builds it in one pass over the keys in increasing order, and depths are assigned as nodes are attached. A node at depth $d$ in a random BST plays the role of a particle of a branching random walk at position $\log$(subtree size), and the height is its maximum: hence Bramson's $\frac32$, in the form of Reed's $\beta$.

In [7]:
def bst_height(n):
    t = rng.permutation(n)            # t[i] = insertion time of key i
    parent = np.full(n, -1); stack = []
    for i in range(n):
        last = -1
        while stack and t[stack[-1]] > t[i]: last = stack.pop()
        if last != -1: parent[last] = i
        if stack: parent[i] = stack[-1]
        stack.append(i)
    order = np.argsort(t); depth = np.zeros(n, dtype=int)
    for v in order:                   # parents are inserted before their children
        if parent[v] >= 0: depth[v] = depth[parent[v]] + 1
    return depth.max()

from scipy.optimize import brentq
alpha = brentq(lambda a: a * math.log(2 * math.e / a) - 1, 2.5, 6); beta = 3 / (2 * math.log(alpha / 2))
print(f"alpha = {alpha:.5f}, beta = {beta:.5f}")
rows = []
for k in range(10, 19, 2):
    n = 2**k; reps = 40 if k <= 14 else 12
    H = np.mean([bst_height(n) for _ in range(reps)]); rows.append((n, H))
L = np.array([math.log(n) for n, _ in rows]); H = np.array([h for _, h in rows])
for name, b in (("Reed beta", beta), ("beta/3", beta / 3)):
    pred = alpha * L - b * np.log(L); c = np.mean(H - pred)
    print(name, " residuals:", np.round(H - pred - c, 2))
for (n, h) in rows: print(f"n = 2^{int(round(math.log2(n))):>2}:  mean height {h:6.2f}   alpha log n = {alpha * math.log(n):6.2f}")

alpha = 4.31107, beta = 1.95303


Reed beta  residuals: [ 0.4  -0.24  0.16  0.09 -0.41]
beta/3  residuals: [ 0.81 -0.07  0.13 -0.11 -0.76]
n = 2^10:  mean height  21.55   alpha log n =  29.88
n = 2^12:  mean height  26.52   alpha log n =  35.86
n = 2^14:  mean height  32.60   alpha log n =  41.83
n = 2^16:  mean height  38.25   alpha log n =  47.81
n = 2^18:  mean height  43.50   alpha log n =  53.79


The mean height is far below $\alpha\log n$ at these sizes, by $8$ to $10$ levels, and most of the gap is the Reed correction. With $\beta\log\log n$ the residuals stay within $\pm0.4$ of a constant, sampling noise with a dozen trees per size; with a third of it, the correction one would get for independent depths, they drift from $+0.8$ to $-0.8$. Devroye's constant $\alpha$ is exactly the speed of a branching random walk: the depth of a key in the tree is a sum of independent "logarithmic splits", and $\alpha$ solves the same large-deviation equation as $c^*=\sqrt{2\log2}$ does for Gaussian steps. This matters in practice: the recursion depth of randomised quicksort is the height of a random binary search tree, about $4.3\log n$ minus Reed's correction, not the $2\log n$ of its average depth.